# 水果和蔬菜价格数据处理

## 下载压缩文件


### ChatGPT  

::: {.callout-tip}
### 提示词

目的：自动从网页端下载 .zip 数据文件

1. 网址：https://www.ers.usda.gov/data-products/fruit-and-vegetable-prices
2. 网页源文件：view-source:https://www.ers.usda.gov/data-products/fruit-and-vegetable-prices
3. 需要下载的文件：vegetables-####.zip, fruit-###.zip
   - 规则：包含 `vegetables` 或 `fruit` 关键词，且后缀为 `.zip` 的所有文件
   - note: 我通过右击下载链接的方式，发现 **fruit-2013.zip** 对应的下载网址是：`https://ers.usda.gov/sites/default/files/_laserfiche/DataFiles/51035/fruit-2013.zip?v=41740`
4. 目标文件夹： 'D:\Github\ds_data\data\Fruit_and_Vegetable_Prices\data_raw'
5. 语言：Python 
   
:::

- 文件清单：
  - `vegetables-2013.zip`, `vegetables-2016.zip`, `vegetables-2020.zip`, 
  - `fruit-2013.zip`, `fruit-2016.zip`, `fruit-2020.zip`

In [9]:
import requests
import re
import os
from urllib.parse import urljoin

# 设置 URL 和目标路径
base_url = "https://www.ers.usda.gov/data-products/fruit-and-vegetable-prices"
target_folder = r"D:\Github\ds_data\data\Fruit_and_Vegetable_Prices\data_raw"

# 创建目标路径
os.makedirs(target_folder, exist_ok=True)

# 获取网页 HTML
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}
response = requests.get(base_url, headers=headers)
html_text = response.text

# 用正则表达式匹配 .zip 链接（fruit 或 vegetables）
pattern = r'href="(.*?(fruit|vegetables)[^"]*?\.zip[^"]*?)"'
matches = re.findall(pattern, html_text, flags=re.IGNORECASE)

# 构造完整链接
zip_links = [urljoin(base_url, m[0]) for m in matches]

print(f"共找到 {len(zip_links)} 个 .zip 文件链接。")

# 下载文件
for link in zip_links:
    filename = os.path.basename(link.split("?")[0])  # 去掉 ?v=xxx 参数
    save_path = os.path.join(target_folder, filename)
    print(f"正在下载：{filename} ...")

    try:
        with requests.get(link, stream=True) as r:
            r.raise_for_status()
            with open(save_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        print(f"下载完成：{filename}")
    except Exception as e:
        print(f"下载失败：{filename}，原因：{e}")


共找到 6 个 .zip 文件链接。
正在下载：fruit-2013.zip ...
下载完成：fruit-2013.zip
正在下载：fruit-2016.zip ...
下载完成：fruit-2016.zip
正在下载：fruit-2020.zip ...
下载完成：fruit-2020.zip
正在下载：vegetables-2013.zip ...
下载完成：vegetables-2013.zip
正在下载：vegetables-2016.zip ...
下载完成：vegetables-2016.zip
正在下载：vegetables-2020.zip ...
下载完成：vegetables-2020.zip


In [20]:
import os
os.chdir(target_folder)
print(os.getcwd())
print(os.listdir(target_folder))

D:\Github\ds_data\data\Fruit_and_Vegetable_Prices\data_raw
['copy', 'downloader.log', 'fruit-2013.zip', 'fruit-2016.zip', 'fruit-2020.zip', 'vegetables-2013.zip', 'vegetables-2016.zip', 'vegetables-2020.zip']


### 豆包

> [豆包问答过程](https://www.doubao.com/thread/w24fdaaf190d3f71a)

- 我用豆包只是为了测试我的提示词是否合理，现已达到目标，因此，下面的代码无需重复执行了。

**Round 1** 提示词发送后，豆包给的代码可以运行，但无法下载文件。不过豆包自动生成了一个日志文件，内容如下：

```txt
2025-05-20 00:54:22,003 - INFO - 开始爬取下载链接...
2025-05-20 00:54:24,145 - WARNING - 未找到符合条件的下载链接
```

我把上述内容作为 **Round 2** 的提示词反馈给豆包，它这次返回的代码可以正常下载数据了。 

::: {.callout-tip}
### 提示词

(Round 1)：

目的：自动从网页端下载 .zip 数据文件

1. 网址：https://www.ers.usda.gov/data-products/fruit-and-vegetable-prices
2. 网页源文件：view-source:https://www.ers.usda.gov/data-products/fruit-and-vegetable-prices
3. 需要下载的文件：vegetables-####.zip, fruit-###.zip
   - 规则：包含 `vegetables` 或 `fruit` 关键词，且后缀为 `.zip` 的所有文件
   - note: 我通过右击下载链接的方式，发现 **fruit-2013.zip** 对应的下载网址是：`https://ers.usda.gov/sites/default/files/_laserfiche/DataFiles/51035/fruit-2013.zip?v=41740`
4. 目标文件夹： 'D:\Github\ds_data\data\Fruit_and_Vegetable_Prices\data_raw'
5. 语言：Python 

(Round 2)：

2025-05-20 00:54:22,003 - INFO - 开始爬取下载链接...    
2025-05-20 00:54:24,145 - WARNING - 未找到符合条件的下载链接
:::

以下是豆包返回的代码：

In [11]:
# 豆包

import os
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import logging
from datetime import datetime

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='downloader.log'
)
logger = logging.getLogger(__name__)

def get_links(url):
    """提取网页中符合规则的.zip文件链接（包括文本中的隐藏链接）"""
    try:
        response = requests.get(url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        logger.error(f"无法访问网页: {e}")
        return []
    
    soup = BeautifulSoup(response.text, 'html.parser')
    base_url = urljoin(url, '/')  # 基础URL
    links = []
    
    # 匹配文本中的链接模式（如 Archived 2013 Data Tables for Fruit）
    pattern = r'Archived\s+(\d{4})\s+Data Tables for\s+(Fruit|Vegetables)'
    matches = re.findall(pattern, soup.get_text(), re.IGNORECASE)
    
    for year, category in matches:
        # 构造.zip文件的URL（根据已知格式推断）
        # 示例：https://ers.usda.gov/sites/default/files/_laserfiche/DataFiles/51035/fruit-2013.zip
        file_type = category.lower()
        url_suffix = f"{file_type}-{year}.zip"
        # 注意：此处URL路径可能需要根据实际情况调整，当前为示例路径
        zip_url = f"https://ers.usda.gov/sites/default/files/_laserfiche/DataFiles/51035/{url_suffix}"
        
        # 验证链接格式（包含fruit/vegetables和.zip）
        if (file_type in ['fruit', 'vegetables'] and 
            zip_url.lower().endswith('.zip')):
            links.append(zip_url)
    
    return links

def download_file(url, target_folder):
    # 保持原有逻辑不变（见上文）
    os.makedirs(target_folder, exist_ok=True)
    filename = os.path.basename(urlparse(url).path)
    local_path = os.path.join(target_folder, filename)
    
    if os.path.exists(local_path):
        logger.info(f"文件已存在: {filename}")
        return local_path
    
    try:
        logger.info(f"开始下载: {filename}")
        response = requests.get(url, stream=True)
        response.raise_for_status()
        
        with open(local_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        logger.info(f"下载完成: {filename}")
        return local_path
    except Exception as e:
        logger.error(f"下载失败: {filename}, 错误: {e}")
        return None

if __name__ == "__main__":
    url = "https://www.ers.usda.gov/data-products/fruit-and-vegetable-prices"
    target_folder = r"D:\Github\ds_data\data\Fruit_and_Vegetable_Prices\data_raw"
    
    logger.info("开始爬取下载链接...")
    links = get_links(url)
    
    if not links:
        logger.warning("未找到符合条件的下载链接")
    else:
        logger.info(f"找到 {len(links)} 个符合条件的下载链接: {links}")
        for link in links:
            download_file(link, target_folder)
    
    logger.info("下载任务完成")

## 解压缩包

::: {.callout-tip}
### 提示词

- 把以 `fruit` 开头的 .zip 文件解压到 [fruit] 文件夹中
- 把以 `vegetables` 开头的 zip 文件解压到 [vegetables] 文件夹中
- 完成后，统计上述两个新生成的子文件夹内的文件数量
  
:::

In [21]:
import zipfile

# 创建目标子文件夹
fruit_folder = os.path.join(target_folder, "fruit")
vegetables_folder = os.path.join(target_folder, "vegetables")
os.makedirs(fruit_folder, exist_ok=True)
os.makedirs(vegetables_folder, exist_ok=True)

# 遍历 target_folder 下的所有 zip 文件并解压
for filename in os.listdir(target_folder):
    if filename.lower().endswith(".zip"):
        zip_path = os.path.join(target_folder, filename)
        if filename.startswith("fruit"):
            extract_path = fruit_folder
        elif filename.startswith("vegetables"):
            extract_path = vegetables_folder
        else:
            continue
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)

# 统计文件数量
fruit_files = [f for f in os.listdir(fruit_folder) if os.path.isfile(os.path.join(fruit_folder, f))]
vegetables_files = [f for f in os.listdir(vegetables_folder) if os.path.isfile(os.path.join(vegetables_folder, f))]

print(f"fruit 文件夹内文件数量: {len(fruit_files)}")
print(f"vegetables 文件夹内文件数量: {len(vegetables_files)}")

fruit 文件夹内文件数量: 117
vegetables 文件夹内文件数量: 177


## 清理重复文件、统一命名

[fruit] 文件夹下的前 8 个文件列举如下：
  
   ![](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250520012922.png)

仔细对比后发现：

- 2016 年数据的有两个文件：一个是 `.csv` 格式的，另一个是 `.xlsx` 格式的。对比文档内容后发现，两份数据文件的内容完全相同。进一步抽查了其十种水果在 2016 年的数据，发现都存在上述规律。
  - 最终决定：删除 [fruit] 文件夹下的所有 `.csv` 文件，保留 `.xlsx` 文件，以便后续采用循环语句进行统一处理。 
- 为了保持各年度上的文件命名方式统一，做如下处理：
  - 将 [fruit] 文件夹下的所有文件名中的 ` ` (空格) 替换为 `_` (下划线)；
  - 将文件名中的有英文单词替换为小写字母。

Note: [vegetables] 文件夹下也存在相同的问题，因此，采用相同的处理方案。

::: {.callout-tip}
### 提示词

- 目标文件夹：
  - {fruit_folder} = '.\Fruit_and_Vegetable_Prices\data_raw\fruit'
  - {vegetables_folder} = '.\Fruit_and_Vegetable_Prices\data_raw\vegetables'
- 处理：
  - 删除目标文件夹下的所有 `.csv` 文件
  - 将文件名中的 ` ` (空格) 替换为 `_` (下划线)；
  - 将文件名中的英文单词替换为小写字母。
:::

In [ ]:
# 删除 fruit_folder 下所有 .csv 文件，并规范化文件名
for fname in os.listdir(fruit_folder):
    old_path = os.path.join(fruit_folder, fname)
    # 只处理文件
    if os.path.isfile(old_path):
        # 删除 .csv 文件
        if fname.lower().endswith('.csv'):
            os.remove(old_path)
            continue
        # 替换空格为下划线，文件名转小写
        new_fname = fname.replace(" ", "_").lower()
        new_path = os.path.join(fruit_folder, new_fname)
        if old_path != new_path:
            os.rename(old_path, new_path)

## 导入 excel 数据文件

### 思路

这里以 [fruit] 文件夹下的文件为例进行说明，[vegetables] 文件夹下的文件也采用相同的处理方式。

- 目标文件夹：`{fruit_folder}`
- 文件数量：117 个 `.xlsx` 文件
- 用循环语句导入这些 Excel 文件，采用纵向合并 (`merge`) 的方式合并为一个数据框。
  - 首先，我们需要分析这些 Excel 文件的内部结构，找出规律，以便采用循环语句进行统一导入。
    - 如果数据结构是非结构化的，则需要进行结构化处理，以便转换成「整洁数据」
    - 如果不同的 Excel 文档的内部结构有差别，则需要分类处理，或编写能够处理不同结构的具有「包容性」的代码
  - 其次，将上述导入单个 Excel 文档的代码封装成函数，以便使循环语句的代码更简洁
  - 最后，采用 `pd.concat()` 函数将所有数据框进行纵向合并

### 分析 .xlsx 文档的特征 1：找规律

由于文件的数量不多 (117 个)，我抽取了一些 EXcel 文档，尝试找出规律。
   
   ![](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250520013804.png)

   ![](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250520092820.png)

   ![](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250520091552.png)

   ![](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250520092543.png)

   ![](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250520084449.png)

我们需要把上述半结构化的数据转换成整洁数据。浏览网页，发现 [ALL FRUITS – Average prices (CSV format)](https://ers.usda.gov/sites/default/files/_laserfiche/DataFiles/51035/Fruit-Prices-2022.csv?v=74889) 页面已经有了整洁数据的格式。该数据文件的格式为：

一种可能的目标格式如下：

 ![](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250520083653.png)

- 观察到 csv 文件的第一行是表头，第二行是数据类型，第三行是数据内容
- 因此，读取 csv 文件时，指定 `header=2`，并且 `skiprows=3`，这样就可以直接读取数据了
- 读取数据时，指定 `usecols` 参数，选择需要的列

明白了。你说得对，这份数据的结构本质上不是“按省份展开的跨列宽表”，而是**一份竖向类别嵌套 + 横向成对指标的宽表格**。下面我将重新组织讲义内容，更符合你目前的教学目标。

---

### 分析 .xlsx 文档的特征 2：单个 EXcel 文档的处理

经过上面的分析，发现 `pineapple_2020.xlsx` 文档的结构最具代表性。我们可以先对它进行处理，然后再对其他文档进行类似的处理。该文档的数据结构如下：

![pineapple_2020.xlsx 文档的数据结构](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250520233829.png)


#### 原始数据的结构与问题

观察这张表格，可以发现它存在多个「非整洁」的结构问题：

- 分类信息混合: [1a] 和 [1b]
  - 不同级别的类目信息混放在一列中。第一列本应表示产品的形态（如 Fresh、Canned、Juice 等），但在实际表格中，**一级分类（如 Form）与各子类别（如 Canned-Packed in juice2）混在一起**，既非标准变量名，也非观测值。

- 多指标交叉在一行中: [2a] 和 [2b]

  - 数据从第 2 列开始，分别包含多个指标，但这些指标**成对出现**，例如：
    - 第 2 列是价格（例如 $1.38），第 3 列是单位（如 per lb）
    - 第 5 列是 “Size of a cup equivalent (每杯的可食用重量)”，第 6 列是“数量单位”

- 第 11-17 行的注释信息 ([3]) 可以删除，随后记录到数据字典或数据文件的 readme 文件中。

- 缺少合法变量名

  - 原始列没有标准的变量名，不便后续引用。例如：第 2 列、第 3 列应该命名为 `retail_price`, `unit`，第五列和第六列也需要分别命名。

#### 整洁数据的目标格式

我们希望将原始数据转化为「整洁数据」（tidy data），其结构应如下所示：

| form                       | retail\_price | unit   | yield\_factor | size\_of\_a\_cup\_equivalent | quality | price\_per\_cup\_equivalent |
| -------------------------- | ------------- | ------ | ------------- | ---------------------------- | ------- | --------------------------- |
| Fresh                      | 1.38          | per lb | 0.55          | 0.61                         | 0.75    | 2.51                        |
| Canned-Packed in juice2    | 1.32          | per lb | 0.91          | 0.85                         | 0.93    | 1.42                        |
| Canned-Packed in syrup or water | 1.23          | per lb | 0.94          | 0.87                         | 0.95    | 1.35                        |
| Dried                      | 5.65          | per lb | 0.99          | 1.07                         | 0.99    | 5.62                        |
| Juice-Ready to drink       | 2.45          | per qt | 0.99          | 0.93                         | 0.99    | 2.63                        |
| Juice-Frozen               | 2.15          | per qt | 0.99          | 0.97                         | 0.99    | 2.21                        |

#### 对应字段命名如下：

| 原始列位置 | 新列名                        | 含义说明          |
| ----- | -------------------------- | ------------- |
| 第 1 列 | `form`                     | 产品形态（含分类子类）   |
| 第 2 列 | `retail_price`             | 零售价格          |
| 第 3 列 | `unit`                     | 价格单位，如 per lb |
| 第 4 列 | `yield_factor`             | 转换系数（如净重占比）   |
| 第 5 列 | `size_of_a_cup_equivalent` | 每杯的可食用重量      |
| 第 6 列 | `quality`                  | 数量    |
| 第 7 列 | `price_per_cup_equivalent` | 单杯对应价格（标准化比较） |


#### 清洗与转换的建议流程

以下是建议的数据清洗步骤：

1. **读取原始 Excel 表格**；
2. **跳过前两行**（说明行、原变量名）；
3. **处理第一列的分类结构**：
   * 使用 **向下填充** 方式，将 `Form` 替代为空的上层分类；
   * 将 `Form` 与下属分类合并为一列，如 “Canned-Packed in juice2”；
     * Note: 随后将 [fruit] 文件夹下的所有 EXcel 文档总行合并后，再统一将变量名中的 ` ` (空格) 替换为 `_` (下划线)，并将英文单词替换为小写字母。
4. **为每列添加标准变量名**（如上表所列）；
5. **去除注释行**：首字符为数字 ('\d') 或第一个单词为 'Source:'；
6. **保存为整洁数据框，便于后续分析和可视化**。



::: {callout-tip}
### 提示词

以 [fruit] 文件夹下的 pipeapple_2020.xlsx 文件为例来说明数据清洗过程。因此，需要按如下步骤 clean 导入后的数据：

1. 文件名中下划线前的的第一个单词 'pipeapple' 作为数据框的第一列: 'fruit'
2. 文件名中下划线后的数字 '2020' 作为数据框的第二列: 'year'
3. 删除表中第 1-2 行的数据
4. 删除表中首字符为数字 ('\d') 的行
6. 第一列既包含了一级分类名称又包含了二级分类名称，需要酌情合并。最终，第一列应该是：
Form
Fresh
Canned-Packed in juice2
Canned-Packed in syrup or water 
Dried
Juice-Ready to drink
Juice-Frozen
7. 变量名：才
   第一列：form；第二列：retail\_price；第三列：


:::

这样处理不行，有些表格不是以 'Juice' 开头的，我还有很多表格需要处理。要采用一个更一般化的处理方法。1. 如果第 i 行的首字母不是 ' '，则分析第 i+j (j>0) 行的首字符是否为 ' '，若是，则将第 i 行的字符 (stri_i) 与第 i+j 行的去除了左侧空格后的字符 (str_j) 进行拼接，拼接为 'str_i-str_j'

5. 若第一列的首字符为空格 (' ')，则将该列中的字符串放入一个新变量 'category' 中

In [76]:
import pandas as pd
import re
import os

# 构造文件路径
fruit_folder = os.path.join(target_folder, "fruit")
filename = "pineapple_2020.xlsx"
file_path = os.path.join(fruit_folder, filename)

# 提取 fruit 和 year
match = re.match(r"([a-zA-Z]+)_(\d{4})", filename)
fruit = match.group(1).lower() if match else ""
year = int(match.group(2)) if match else None

# 读取 Excel，保留第一列空格，跳过首行
df = pd.read_excel(file_path, header=1, dtype=str, keep_default_na=False)

# 呈现数据框
df

,Form,Average retail price,Unnamed: 2,Preparation yield factor,Size of a cup equivalent,Unnamed: 5,Average price per cup equivalent
0,Fresh1,0.568546746679824,per pound,0.51,0.363762732605048,pounds,0.40552180036463287
1,Canned,,,,,,
2,Packed in juice2,1.43438779242581,per pound,1,0.54013254235295,pounds,0.7747595250429883
3,Packed in syrup or water3,1.40674775087195,per pound,0.65,0.44092452436975516,pounds,0.9542608968637497
4,Dried4,6.64922811752718,per pound,1,0.1543235835294143,pounds,1.026132710801336
5,Juice,,,,,,
6,Ready to drink5,1.02880152764246,per pint,1,8,fl. oz.,0.51440076382123
7,Frozen6,0.697265784588798,per pint,1,8,fl. oz.,0.348632892294399
8,1The USDA National Nutrient Database for Stand...,,,,,,
9,2Consumers are assumed to eat the solid fruit ...,,,,,,


In [77]:
# 删除注释文字：首字符为数字或第一个单词为 'Source:' 的行
df = df[
    ~df.iloc[:, 0].astype(str).str.match(r"^\d") &
    ~df.iloc[:, 0].astype(str).str.strip().str.startswith("Source:")
].reset_index(drop=True)

df

,Form,Average retail price,Unnamed: 2,Preparation yield factor,Size of a cup equivalent,Unnamed: 5,Average price per cup equivalent
0,Fresh1,0.568546746679824,per pound,0.51,0.363762732605048,pounds,0.40552180036463287
1,Canned,,,,,,
2,Packed in juice2,1.43438779242581,per pound,1,0.54013254235295,pounds,0.7747595250429883
3,Packed in syrup or water3,1.40674775087195,per pound,0.65,0.44092452436975516,pounds,0.9542608968637497
4,Dried4,6.64922811752718,per pound,1,0.1543235835294143,pounds,1.026132710801336
5,Juice,,,,,,
6,Ready to drink5,1.02880152764246,per pint,1,8,fl. oz.,0.51440076382123
7,Frozen6,0.697265784588798,per pint,1,8,fl. oz.,0.348632892294399


In [79]:
# 新增 with_subcat 列：第二列为空字符串则为1，否则为0
df["with_subcat"] = (df.iloc[:, 1] == "").astype(int)
df2 = df[["Form", "with_subcat"]].copy()
df2

,Form,with_subcat
0,Fresh1,0
1,Canned,0
2,Packed in juice2,0
3,Packed in syrup or water3,0
4,Dried4,0
5,Juice,1
6,Ready to drink5,0
7,Frozen6,0


In [ ]:
# 判断第二列是否全为空字符串
with_subcat = (df.iloc[:, 1] == "").all()


with_subcat = False


1. 在 df 中新增一个变量: with_subcat
2. 如果 df 的第二列为 "", 则该行的 with_subcat = 1，否则 with_subcat = 0

In [ ]:



# 更通用的 form 分类处理
form_col = df.iloc[:, 0].copy()
filled_form = []
i = 0
n = len(form_col)
while i < n:
    val = form_col.iloc[i]
    if isinstance(val, str) and not val.startswith(" "):
        # 检查后续是否有以空格开头的子项
        j = 1
        has_child = False
        while i + j < n:
            next_val = form_col.iloc[i + j]
            if isinstance(next_val, str) and next_val.startswith(" "):
                # 拼接父项和子项
                filled_form.append(f"{val.strip()}-{next_val.strip()}")
                has_child = True
                j += 1
            else:
                break
        if not has_child:
            filled_form.append(val.strip())
        i += j if has_child else 1
    elif isinstance(val, str) and val.startswith(" "):
        # 如果是孤立的子项，直接去除空格
        filled_form.append(val.strip())
        i += 1
    else:
        filled_form.append(val)
        i += 1
# 补齐长度（如果有多余的行）
while len(filled_form) < n:
    filled_form.append(None)
df.iloc[:, 0] = filled_form

# 重命名列
df.columns = [
    "form", "retail_price", "unit", "yield_factor",
    "size_of_a_cup_equivalent", "quality", "price_per_cup_equivalent"
]

# 添加 fruit 和 year 列
df.insert(0, "fruit", fruit)
df.insert(1, "year", year)

df = df.reset_index(drop=True)
df.head(10)


::: {.callout-tip}
### 提示词

![pineapple_2020.xlsx 文档的数据结构](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250520233829.png)

目的：将 [fruit] 文件夹下 `pipeapple_2020.xlsx` 文件导入并转换为整洁数据：

1. 导入数据时，务必维持第一列的原始字符状态，切勿删除第一列的行首的空格；
2. ![](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250520173207.png)
4. 第一列既包含了一级分类名称又包含了二级分类名称，需要酌情合并。最终，第一列应该是：
Form
Fresh
Canned-Packed in juice2
Canned-Packed in syrup or water 
Dried
Juice-Ready to drink
Juice-Frozen 
1. 文件名中下划线前的的第一个单词 'pipeapple' 作为数据框的第一列: 'fruit'
2. 文件名中下划线后的数字 '2020' 作为数据框的第二列: 'year'
3. 删除表中第 1 行的数据
4. 删除表中首字符为数字 ('\d') 或第一个单词为 'Source:' 的行 (表格的注释内容)
5. 变量名：
   第一列：`form`；
   第二列：`retail_price`；
   第三列：`unit`；
   第四列：`yield_factor`；
   第五列：`size_of_a_cup_equivalent`；
   第六列：`quality`；
   第七列：`price_per_cup_equivalent`；

:::   

3. 使用 **向下填充** 方式，将 `Form` 替代为空的上层分类；

In [64]:
import pandas as pd
import re
import os

# 构造文件路径
fruit_folder = os.path.join(target_folder, "fruit")
filename = "pineapple_2020.xlsx"
file_path = os.path.join(fruit_folder, filename)

# 提取 fruit 和 year
match = re.match(r"([a-zA-Z]+)_(\d{4})", filename)
fruit = match.group(1).lower() if match else ""
year = int(match.group(2)) if match else None

# 读取 Excel，保留第一列空格，跳过首行
df = pd.read_excel(file_path, header=1, dtype=str, keep_default_na=False)

# 删除首字符为数字或第一个单词为 'Source:' 的行
df = df[
    ~df.iloc[:, 0].astype(str).str.match(r"^\d") &
    ~df.iloc[:, 0].astype(str).str.strip().str.startswith("Source:")
].reset_index(drop=True)

# 打印时左对齐，确保空格被正确显示
for val in df["Form"]:
    print(f"'{val}'")


'Fresh1'
'Canned'
'  Packed in juice2'
'  Packed in syrup or water3'
'Dried4'
'Juice'
'Ready to drink5'
'Frozen6'


In [ ]:

# 新算法：分离二级分类到 son_form，并下填 form
df["son_form"] = ""  # 新增 son_form 列

for idx in df.index:
    val = str(df.iloc[idx, 0])
    if val.startswith(" "):  # 二级项
        df.at[idx, "son_form"] = val.strip()
        df.at[idx, df.columns[0]] = ""  # form 置空



# form 列向下填充
df[df.columns[0]] = df[df.columns[0]].replace("", pd.NA).ffill()

# 删除 son_form 为空的行（即只保留二级项）
df = df[df["son_form"] != ""].reset_index(drop=True)

# 处理列名：去除特殊符号和空格，统一小写
df.columns = (
    df.columns.str.strip()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# 检查实际列名，选择正确的数值列名
print("列名:", list(df.columns))

# 你需要根据实际表头调整数值列名，以下是常见的列名映射
possible_numeric_columns = [
    "retail_price", "yield_factor", "price_per_cup_equivalent",
    "average_retail_price", "preparation_yield_factor", "average_price_per_cup_equivalent"
]
numeric_columns = [col for col in possible_numeric_columns if col in df.columns]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce").round(2)  # 转换为数值并保留两位小数

# 处理单位列（可选：若需拆分单位与数值）
# df[['average_retail_price_value', 'average_retail_price_unit']] = df['average_retail_price'].str.split(' ', expand=True)

# 去除全空行
df = df.dropna(how='all').reset_index(drop=True)


# 保存为新文件
output_filename = f"cleaned_{fruit}_{year}.csv"
output_path = os.path.join(fruit_folder, output_filename)
df.to_csv(output_path, index=False)
print(f"数据已保存至：{output_path}")


# 输出结果（示例：打印前5行）
print(f"清洁后数据（{fruit} - {year}）：")
df.head(10)

0             Fresh1
1             Canned
2                   
3                   
4             Dried4
5              Juice
6    Ready to drink5
7            Frozen6
Name: Form, dtype: object


In [ ]:

# form 列向下填充
df[df.columns[0]] = df[df.columns[0]].replace("", pd.NA).ffill()

# 删除 son_form 为空的行（即只保留二级项）
df = df[df["son_form"] != ""].reset_index(drop=True)

# 处理列名：去除特殊符号和空格，统一小写
df.columns = (
    df.columns.str.strip()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# 检查实际列名，选择正确的数值列名
print("列名:", list(df.columns))

# 你需要根据实际表头调整数值列名，以下是常见的列名映射
possible_numeric_columns = [
    "retail_price", "yield_factor", "price_per_cup_equivalent",
    "average_retail_price", "preparation_yield_factor", "average_price_per_cup_equivalent"
]
numeric_columns = [col for col in possible_numeric_columns if col in df.columns]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce").round(2)  # 转换为数值并保留两位小数

# 处理单位列（可选：若需拆分单位与数值）
# df[['average_retail_price_value', 'average_retail_price_unit']] = df['average_retail_price'].str.split(' ', expand=True)

# 去除全空行
df = df.dropna(how='all').reset_index(drop=True)


# 保存为新文件
output_filename = f"cleaned_{fruit}_{year}.csv"
output_path = os.path.join(fruit_folder, output_filename)
df.to_csv(output_path, index=False)
print(f"数据已保存至：{output_path}")


# 输出结果（示例：打印前5行）
print(f"清洁后数据（{fruit} - {year}）：")
df.head(10)

In [51]:
import pandas as pd
import re
import os

# 第一步：指定文件路径
file = "pineapple_2020.xlsx"
folder = "D:/Github/ds_data/data/Fruit_and_Vegetable_Prices/data_raw/fruit"
filepath = os.path.join(folder, file)

# 第二步：提取 fruit 和 year 信息
filename = os.path.basename(filepath)
match = re.match(r"([a-zA-Z_]+)[\s_](\d{4})", filename)
fruit = match.group(1).lower() if match else "unknown"
year = int(match.group(2)) if match else None

# 第三步：读取 Excel，跳过前两行
df = pd.read_excel(filepath, skiprows=1, header=None)

# 第四步：删除首字符为数字或第一个单词为 'Source:' 的行
df = df[
    ~df.iloc[:, 0].astype(str).str.match(r"^\d") &
    ~df.iloc[:, 0].astype(str).str.strip().str.startswith("Source:")
].reset_index(drop=True)

# 第五步：重新设置列名
df.columns = [
    "Form", "retail_price", "unit", 
    "yield_factor", "cup_size", "cup_unit", "price_per_cup"
]

# 第六步：清洗 form 列，合并分类 + 子项
form_list = []
current_category = None
for val in df["Form"]:
    val_str = str(val)
    if re.match(r"^\s", val_str):  # 子项
        form_list.append(f"{current_category}–{val_str.strip()}")
    else:  # 新分类或独立项
        current_category = val_str.strip()
        form_list.append(current_category)
df["form"] = form_list

# 第七步：清洗价格字段（去除 $ 符号，转换为 float，忽略无法转换的空格或无效值）
df["retail_price"] = (
    df["retail_price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .replace(r"^\s*$", pd.NA, regex=True)
)
df["retail_price"] = pd.to_numeric(df["retail_price"], errors="coerce")

df["price_per_cup"] = (
    df["price_per_cup"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .replace(r"^\s*$", pd.NA, regex=True)
)
df["price_per_cup"] = pd.to_numeric(df["price_per_cup"], errors="coerce")

# 第八步：添加 fruit 和 year 字段
df["fruit"] = fruit
df["year"] = year

# 第九步：重排列顺序，丢弃无效行
df_clean = df[[
    "fruit", "year", "form", "retail_price", "unit", 
    "yield_factor", "cup_size", "cup_unit", "price_per_cup"
]].dropna(subset=["form"]).reset_index(drop=True)

# 第十步：保存为 Excel 文件（可选）
df_clean.to_excel("pineapple_2020_clean.xlsx", index=False)

# 查看结果（可选）
df_clean


,fruit,year,form,retail_price,unit,yield_factor,cup_size,cup_unit,price_per_cup
0,pineapple,2020,Form,NaN,NaN,Preparation yield factor,Size of a cup equivalent,NaN,NaN
1,pineapple,2020,Fresh1,0.568547,per pound,0.51,0.363763,pounds,0.405522
2,pineapple,2020,Canned,NaN,NaN,,NaN,,NaN
3,pineapple,2020,Canned–Packed in juice2,1.434388,per pound,1,0.540133,pounds,0.774760
4,pineapple,2020,Canned–Packed in syrup or water3,1.406748,per pound,0.65,0.440925,pounds,0.954261
5,pineapple,2020,Dried4,6.649228,per pound,1,0.154324,pounds,1.026133
6,pineapple,2020,Juice,NaN,NaN,NaN,NaN,NaN,NaN
7,pineapple,2020,Ready to drink5,1.028802,per pint,1,8,fl. oz.,0.514401
8,pineapple,2020,Frozen6,0.697266,per pint,1,8,fl. oz.,0.348633


In [ ]:
import pandas as pd
import re
import os

# 构造文件路径
fruit_folder = os.path.join(target_folder, "fruit")
filename = "pineapple_2020.xlsx"
file_path = os.path.join(fruit_folder, filename)

# 提取 fruit 和 year
match = re.match(r"([a-zA-Z]+)_(\d{4})", filename)
fruit = match.group(1).lower() if match else ""
year = int(match.group(2)) if match else None

# 读取 Excel，保留第一列空格，跳过首行
df = pd.read_excel(file_path, header=1, dtype=str)

# 删除首字符为数字或第一个单词为 'Source:' 的行
df = df[
    ~df.iloc[:, 0].astype(str).str.match(r"^\d") &
    ~df.iloc[:, 0].astype(str).str.strip().str.startswith("Source:")
].reset_index(drop=True)

# 处理 category 列
category = None
if df.iloc[0, 0].startswith(" "):
    category = df.iloc[0, 0].strip()
    df = df.iloc[1:].reset_index(drop=True)
    df["category"] = category
else:
    df["category"] = None

# 重命名列
df.columns = [
    "form", "retail_price", "unit", "yield_factor",
    "size_of_a_cup_equivalent", "quality", "price_per_cup_equivalent", "category"
]

# 添加 fruit 和 year 列
df.insert(0, "fruit", fruit)
df.insert(1, "year", year)

df.head()

In [ ]:
import pandas as pd
import re
import os

# 第一步：指定文件路径
file = "pineapple_2020.xlsx"
folder = "D:/Github/ds_data/data/Fruit_and_Vegetable_Prices/data_raw/fruit"
filepath = os.path.join(folder, file)

# 第二步：提取 fruit 和 year 信息
filename = os.path.basename(filepath)
match = re.match(r"([a-zA-Z_]+)[\s_](\d{4})", filename)
fruit = match.group(1).lower() if match else "unknown"
year = int(match.group(2)) if match else None

# 第三步：读取 Excel，跳过前两行
df = pd.read_excel(filepath, skiprows=2, header=None)

# 第四步：删除首字符为数字的行（无效脚注）
df = df[df[0].astype(str).str.match(r"^\D")]

# 第五步：重新设置列名
df.columns = [
    "Form", "retail_price", "unit", 
    "yield_factor", "cup_size", "cup_unit", "price_per_cup"
]

# 第六步：清洗 form 列，合并分类 + 子项
form_list = []
current_category = None
for val in df["Form"]:
    val_str = str(val)
    if re.match(r"^\s", val_str):  # 子项
        form_list.append(f"{current_category}–{val_str.strip()}")
    else:  # 新分类或独立项
        current_category = val_str.strip()
        form_list.append(current_category)
df["form"] = form_list

# 第七步：清洗价格字段（去除 $ 符号，转换为 float）
df["retail_price"] = df["retail_price"].astype(str).str.replace("$", "", regex=False).astype(float)
df["price_per_cup"] = df["price_per_cup"].astype(str).str.replace("$", "", regex=False).astype(float)

# 第八步：添加 fruit 和 year 字段
df["fruit"] = fruit
df["year"] = year

# 第九步：重排列顺序，丢弃无效行
df_clean = df[[
    "fruit", "year", "form", "retail_price", "unit", 
    "yield_factor", "cup_size", "cup_unit", "price_per_cup"
]].dropna(subset=["form"]).reset_index(drop=True)

# 第十步：保存为 Excel 文件（可选）
df_clean.to_excel("pineapple_2020_clean.xlsx", index=False)

# 查看结果（可选）
print(df_clean.head())



![20250520173207](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250520173207.png)

In [38]:
import pandas as pd
import re
import os

# 构造文件路径
fruit_folder = os.path.join(target_folder, "fruit")
filename = "pineapple_2020.xlsx"
file_path = os.path.join(fruit_folder, filename)

# 提取 fruit 和 year
match = re.match(r"([a-zA-Z]+)_(\d{4})", filename)
fruit = match.group(1).lower() if match else ""
year = int(match.group(2)) if match else None

# 读取 Excel，保留第一列空格，跳过首行
df = pd.read_excel(file_path, header=1, dtype=str)
df


,Form,Average retail price,Unnamed: 2,Preparation yield factor,Size of a cup equivalent,Unnamed: 5,Average price per cup equivalent
0,Fresh1,0.568546746679824,per pound,0.51,0.363762732605048,pounds,0.40552180036463287
1,Canned,,NaN,,NaN,,NaN
2,Packed in juice2,1.43438779242581,per pound,1,0.54013254235295,pounds,0.7747595250429883
3,Packed in syrup or water3,1.40674775087195,per pound,0.65,0.44092452436975516,pounds,0.9542608968637497
4,Dried4,6.64922811752718,per pound,1,0.1543235835294143,pounds,1.026132710801336
5,Juice,NaN,NaN,NaN,NaN,NaN,NaN
6,Ready to drink5,1.02880152764246,per pint,1,8,fl. oz.,0.51440076382123
7,Frozen6,0.697265784588798,per pint,1,8,fl. oz.,0.348632892294399
8,1The USDA National Nutrient Database for Stand...,NaN,NaN,NaN,NaN,NaN,NaN
9,2Consumers are assumed to eat the solid fruit ...,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:

# 删除首字符为数字或第一个单词为 'Source:' 的行
df = df[
    ~df.iloc[:, 0].astype(str).str.match(r"^\d") &
    ~df.iloc[:, 0].astype(str).str.strip().str.startswith("Source:")
].reset_index(drop=True)

df


,Form,Average retail price,Unnamed: 2,Preparation yield factor,Size of a cup equivalent,Unnamed: 5,Average price per cup equivalent
0,Fresh1,0.568546746679824,per pound,0.51,0.363762732605048,pounds,0.40552180036463287
1,Canned,,NaN,,NaN,,NaN
2,Packed in juice2,1.43438779242581,per pound,1,0.54013254235295,pounds,0.7747595250429883
3,Packed in syrup or water3,1.40674775087195,per pound,0.65,0.44092452436975516,pounds,0.9542608968637497
4,Dried4,6.64922811752718,per pound,1,0.1543235835294143,pounds,1.026132710801336
5,Juice,NaN,NaN,NaN,NaN,NaN,NaN
6,Ready to drink5,1.02880152764246,per pint,1,8,fl. oz.,0.51440076382123
7,Frozen6,0.697265784588798,per pint,1,8,fl. oz.,0.348632892294399


In [40]:

# 处理 category 列
category = None
if df.iloc[0, 0].startswith(" "):
    category = df.iloc[0, 0].strip()
    df = df.iloc[1:].reset_index(drop=True)
    df["category"] = category
else:
    df["category"] = None

df


,Form,Average retail price,Unnamed: 2,Preparation yield factor,Size of a cup equivalent,Unnamed: 5,Average price per cup equivalent,category
0,Fresh1,0.568546746679824,per pound,0.51,0.363762732605048,pounds,0.40552180036463287,None
1,Canned,,NaN,,NaN,,NaN,None
2,Packed in juice2,1.43438779242581,per pound,1,0.54013254235295,pounds,0.7747595250429883,None
3,Packed in syrup or water3,1.40674775087195,per pound,0.65,0.44092452436975516,pounds,0.9542608968637497,None
4,Dried4,6.64922811752718,per pound,1,0.1543235835294143,pounds,1.026132710801336,None
5,Juice,NaN,NaN,NaN,NaN,NaN,NaN,None
6,Ready to drink5,1.02880152764246,per pint,1,8,fl. oz.,0.51440076382123,None
7,Frozen6,0.697265784588798,per pint,1,8,fl. oz.,0.348632892294399,None


In [ ]:

# 重命名列
df.columns = [
    "form", "retail_price", "unit", "yield_factor",
    "size_of_a_cup_equivalent", "quality", "price_per_cup_equivalent", "category"
]

# 添加 fruit 和 year 列
df.insert(0, "fruit", fruit)
df.insert(1, "year", year)

df.head()

In [32]:
import pandas as pd
import re
import os

# 1. 构造文件路径
fruit_folder = os.path.join(target_folder, "fruit")
filename = "pineapple_2020.xlsx"
file_path = os.path.join(fruit_folder, filename)

# 2. 提取 fruit 和 year
match = re.match(r"([a-zA-Z]+)_(\d{4})", filename)
fruit = match.group(1).lower() if match else ""
year = int(match.group(2)) if match else None

# 3. 读取 Excel，跳过前两行
df = pd.read_excel(file_path, header=1)
df

,Form,Average retail price,Unnamed: 2,Preparation yield factor,Size of a cup equivalent,Unnamed: 5,Average price per cup equivalent
0,Fresh1,0.568547,per pound,0.51,0.363763,pounds,0.405522
1,Canned,,NaN,,NaN,,NaN
2,Packed in juice2,1.434388,per pound,1,0.540133,pounds,0.774760
3,Packed in syrup or water3,1.406748,per pound,0.65,0.440925,pounds,0.954261
4,Dried4,6.649228,per pound,1,0.154324,pounds,1.026133
5,Juice,NaN,NaN,NaN,NaN,NaN,NaN
6,Ready to drink5,1.028802,per pint,1,8.000000,fl. oz.,0.514401
7,Frozen6,0.697266,per pint,1,8.000000,fl. oz.,0.348633
8,1The USDA National Nutrient Database for Stand...,NaN,NaN,NaN,NaN,NaN,NaN
9,2Consumers are assumed to eat the solid fruit ...,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
# 删除首字符为数字或第一个单词为 'Source:' 的行
df = df[
    ~df.iloc[:, 0].astype(str).str.match(r"^\d") &
    ~df.iloc[:, 0].astype(str).str.strip().str.startswith("Source:")
].reset_index(drop=True)
df

,Form,Average retail price,Unnamed: 2,Preparation yield factor,Size of a cup equivalent,Unnamed: 5,Average price per cup equivalent
0,Fresh1,0.568547,per pound,0.51,0.363763,pounds,0.405522
1,Canned,,NaN,,NaN,,NaN
2,Packed in juice2,1.434388,per pound,1,0.540133,pounds,0.774760
3,Packed in syrup or water3,1.406748,per pound,0.65,0.440925,pounds,0.954261
4,Dried4,6.649228,per pound,1,0.154324,pounds,1.026133
5,Juice,NaN,NaN,NaN,NaN,NaN,NaN
6,Ready to drink5,1.028802,per pint,1,8.000000,fl. oz.,0.514401
7,Frozen6,0.697266,per pint,1,8.000000,fl. oz.,0.348633


In [30]:

# 4. 删除首字符为数字的行
df = df[~df.iloc[:, 0].astype(str).str.match(r"^\d")].reset_index(drop=True)
df


,Form,Average retail price,Unnamed: 2,Preparation yield factor,Size of a cup equivalent,Unnamed: 5,Average price per cup equivalent
0,Fresh1,0.568547,per pound,0.51,0.363763,pounds,0.405522
1,Canned,,NaN,,NaN,,NaN
2,Packed in juice2,1.434388,per pound,1,0.540133,pounds,0.774760
3,Packed in syrup or water3,1.406748,per pound,0.65,0.440925,pounds,0.954261
4,Dried4,6.649228,per pound,1,0.154324,pounds,1.026133
5,Juice,NaN,NaN,NaN,NaN,NaN,NaN
6,Ready to drink5,1.028802,per pint,1,8.000000,fl. oz.,0.514401
7,Frozen6,0.697266,per pint,1,8.000000,fl. oz.,0.348633
8,"Source: USDA, Economic Research Service calcul...",NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 5. 处理 category 列
category = None
if df.iloc[0, 0].startswith(" "):
    category = df.iloc[0, 0].strip()
    df = df.iloc[1:].reset_index(drop=True)
    df["category"] = category
else:
    df["category"] = None

# 6. 重命名列
df.columns = [
    "form", "retail_price", "unit", "yield_factor",
    "size_of_a_cup_equivalent", "quality", "price_per_cup_equivalent", "category"
]

# 7. 添加 fruit 和 year 列
df.insert(0, "fruit", fruit)
df.insert(1, "year", year)

df.head()